# STQD6324 Data Management - Assignment 1
## P166248 - Suraya Balqis binti Ab.Ghafar
### Iris Dataset Classification using Spark MLlib

---

### Overview
**Apache Spark MLlib** is used to build and evaluate three classification models on the **Iris dataset**.
This Notebook is generated from Databricks

### **SECTION 1A: Load Iris Dataset**
#### The Iris dataset is loaded using sklearn as a pandas DataFrame.
#### EDA will be performed on pandas directly (more efficient than converting to/from Spark).
#### Spark conversion happens later in Section 2, just before MLlib processing.

In [0]:
# SECTION 1A: Load Iris Dataset as Pandas DataFrame

# Import Libraries
from sklearn.datasets import load_iris
import pandas as pd

# Load Iris dataset from sklearn
iris = load_iris(as_frame=True)

# Combine features and labels into a single pandas DataFrame
# (iris.data contains the feature columns, iris.target is the label column)
iris_df = pd.concat([iris.data, iris.target.rename("label")], axis=1)

print("Iris dataset loaded as pandas DataFrame:")
print(f"Shape: {iris_df.shape[0]} rows × {iris_df.shape[1]} columns")
print(f"\nFirst 5 rows:")
display(iris_df.head())

### **SECTION 1B: Exploratory Data Analysis (EDA)**

Exploratory Data Analysis helps to understand the Iris dataset structure, distributions, and relationships before building models.

#### **What we'll explore:**
* Dataset structure and statistical summary
* Class distribution (balanced vs imbalanced)
* Feature correlations (identify redundant features)
* Key insights for modeling

In [0]:
# SECTION 1B: Exploratory Data Analysis (EDA)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print("="*70)
print("EXPLORATORY DATA ANALYSIS - IRIS DATASET")
print("="*70)

# iris_df is already pandas - no need to convert from Spark

print("\n[1] DATASET OVERVIEW")
print(f"\nColumn Names and Types:")
print(iris_df.dtypes)

print("\n[2] STATISTICAL SUMMARY")
print("="*70)
print(iris_df.describe())

print("\n[3] MISSING VALUES CHECK")
print("="*70)
print(iris_df.isnull().sum())
if iris_df.isnull().sum().sum() == 0:
    print("✓ No missing values found!")

In [0]:
# Class Distribution
print("[4] CLASS DISTRIBUTION")
print("="*70)

class_counts = iris_df['label'].value_counts().sort_index()
print("\nClass counts:")
for label, count in class_counts.items():
    print(f"  Class {label}: {count} samples ({count/len(iris_df)*100:.1f}%)")

# Visualize class distribution
fig, ax = plt.subplots(figsize=(8, 5))
iris_df['label'].value_counts().sort_index().plot(kind='bar', ax=ax, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
ax.set_title('Class Distribution in Iris Dataset', fontsize=14, fontweight='bold')
ax.set_xlabel('Class Label (0=Setosa, 1=Versicolor, 2=Virginica)', fontsize=11)
ax.set_ylabel('Number of Samples', fontsize=11)
ax.set_xticklabels(['Setosa (0)', 'Versicolor (1)', 'Virginica (2)'], rotation=0)
plt.tight_layout()
plt.show()

In [0]:
# Correlation Analysis
print("[5] CORRELATION ANALYSIS")
print("="*70)

# Calculate correlation matrix
corr_matrix = iris_df.corr()
print("\nCorrelation Matrix:")
print(corr_matrix)

# Visualize correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()


#### EDA SUMMARY - KEY INSIGHTS

✓ Dataset Characteristics:
-  150 samples, 4 features, 3 balanced classes (50 each)
-  No missing values or data quality issues
-  Features are numeric and well-scaled

✓ Feature Insights:
- Petal length & width are highly correlated (~0.96)
- Sepal measurements show moderate correlation with petal measurements
- Sepal width has weaker correlations with other features

✓ Data Quality:
- Distributions are relatively normal
- Dataset is ideal for classification tasks

### SECTION 2: Convert to Spark & Data Preprocessing
#### Convert the Iris dataset from pandas DataFrame to Spark for MLlib processing.
#### Features are assembled into a vector and the dataset is prepared for machine learning.

* Spark MLlib classification requires one column for features (must be a vector) and one column for labels
* It doesn't accept multiple separate columns as input
* This section shows how to prepare the data so Spark MLlib can process it

In [0]:
# SECTION 2: Convert to Spark DataFrame & Data Preprocessing

from pyspark.ml.feature import VectorAssembler

# Step 1: Convert pandas DataFrame to Spark DataFrame
print("[1] Converting pandas DataFrame to Spark DataFrame...")
df = spark.createDataFrame(iris_df)

# Step 2: Assemble features into a single vector column
# Define feature columns
feature_cols = ["sepal length (cm)", "sepal width (cm)", "petal length (cm)", "petal width (cm)"]

# Assemble features into a single vector column
# This combines all four numeric measurements into a single "features" vector
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# Apply the transformation to create a new DataFrame with a "features" column required by Spark MLlib
df_assembled = assembler.transform(df)

print("\n[2] Assemble features into a single vector column for MLlib:")
print("    ✓ Features assembled successfully")

print("\n[3] Preprocessed data ready for MLlib:")
display(df_assembled.select("features", "label"))

### **SECTION 3: Split Dataset**
#### The dataset is split into training (70%) and testing (30%) sets for model evaluation.
#### Training set (70%) is used to train the models, letting them learn patterns from the data
#### Testing set (30%) is kept separate to evaluate model performance on unseen data - give unbiased assessment how well the model generalizes

In [0]:
# SECTION 3: Split Dataset
# Split into training (70%) and testing (30%) sets with a fixed seed for reproducibility

train_df, test_df = df_assembled.randomSplit([0.7, 0.3], seed=42)

print("="*60)
print("TRAIN/TEST SPLIT")
print("="*60)
print(f"Training set size: {train_df.count()} samples (70%)")
print(f"Testing set size:  {test_df.count()} samples (30%)")
print("\n✓ Data split completed successfully!")
print("="*60)

In [0]:
# Prepare DataFrames with Delta-compatible column names
# Required for CrossValidator/TrainValidationSplit (Delta Lake doesn't allow spaces or special characters)

from pyspark.sql.functions import col
import os

print("\n" + "="*60)
print("SETUP FOR AUTOMATED HYPERPARAMETER TUNING")
print("="*60)

# Step 1: Create column mapping
column_mapping = {
    "sepal length (cm)": "sepal_length",
    "sepal width (cm)": "sepal_width",
    "petal length (cm)": "petal_length",
    "petal width (cm)": "petal_width"
}

# Step 2: Rename columns in training and test DataFrames
train_df_clean = train_df
test_df_clean = test_df

# Rename columns in train_df_clean and test_df_clean according to column_mapping
for old_name, new_name in column_mapping.items():
    train_df_clean = train_df_clean.withColumnRenamed(old_name, new_name)
    test_df_clean = test_df_clean.withColumnRenamed(old_name, new_name)

print("\n[1] Column names cleaned for Delta compatibility:")
print(f"    Original: {list(column_mapping.keys())}")
print(f"    Cleaned:  {list(column_mapping.values())}")

# Step 3: Create Unity Catalog volume if it doesn't exist
# This ensures a Delta-compatible storage location exists for Spark MLlib's automated hyperparameter tuning (CrossValidator/TrainValidationSplit).
try:
    spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.mlflow_temp")
    print("\n[2] Unity Catalog volume created/verified: workspace.default.mlflow_temp")
except Exception as e:
    print(f"\n[2] Volume creation note: {e}")

# Step 4: Set environment variable for Spark ML temp path
# This sets the environment variable so Spark MLlib knows where to store temporary files during model tuning.
volume_path = "/Volumes/workspace/default/mlflow_temp"
os.environ['SPARKML_TEMP_DFS_PATH'] = volume_path
print(f"[3] Set SPARKML_TEMP_DFS_PATH: {volume_path}")

print("\n✅ Setup complete! Ready for CrossValidator and TrainValidationSplit")
print("="*60)

### **SECTION 4: Random Forest Classifier with Hyperparameter Tuning**

#### **What is Random Forest?**
Random Forest is an ensemble learning method that builds multiple decision trees and combines their predictions to improve accuracy and robustness. Each tree is trained on a random subset of the data and features, which helps reduce overfitting and increases generalization. The final prediction is made by aggregating the predictions (majority vote for classification) from all trees.

**Key characteristics:**
- Handles both classification and regression tasks.
- Reduces variance compared to a single decision tree.
- Automatically handles feature interactions and non-linear relationships.
- Robust to noise and outliers.

#### **Why use Random Forest for the Iris dataset?**
- The Iris dataset is a classic multiclass classification problem with well-separated classes and numeric features.
- Random Forest can capture complex patterns and interactions between features (e.g., petal and sepal measurements).
- It provides high accuracy, is less prone to overfitting than a single tree, and works well even with small datasets like Iris.
- The model can also provide feature importance, helping to interpret which measurements are most useful for classification.

#### **Hyperparameter Tuning Approach:**
* **Manual Grid Search**: Tests all combinations of hyperparameters systematically
* **Validation Split**: Uses 80% of training data for model fitting, 20% for validation
* **Parameters tuned**: `numTrees` (number of trees) and `maxDepth` (tree depth limit)
* **Selection Criterion**: F1 score on validation set

#### Reference: https://spark.apache.org/docs/latest/ml-classification-regression.html#random-forest-classifier

* `numTrees` (number of trees) - number of decision trees. more trees = more computation but potentially better accuracy
* `maxDepth` (tree depth limit) - maximum depth of each tree. more depth = more complex, but there is a risk of overfitting

In [0]:
# SECTION 4: Random Forest Classifier with Hyperparameter Tuning

from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

print("="*60)
print("RANDOM FOREST CLASSIFIER")
print("="*60)

# Step 1: Split training data for validation
# Training set (already 70% of  original data) split into 80% for training, 20% for validation
# 80% is used to fit models with different hyperparameters
# 20% is used to test which hyperparameters work best
# validation set gives an unbiased way to compare different configurations
train_data, val_data = train_df.randomSplit([0.8, 0.2], seed=42)

# Create all evaluators once (used for both grid search and final evaluation)
# These evaluators are reused in all model sections (4, 5, 6, 6B)
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
evaluator_prec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
evaluator_rec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")

# Step 2: Define hyperparameter grid
# We choose [10, 20, 30] for numTrees and [3, 5, 7] for maxDepth as they are common ranges for small/medium datasets like Iris.
# These values provide a balance between model complexity and computational efficiency, and are recommended in Spark MLlib docs:

num_trees_options = [10, 20, 30]
max_depth_options = [3, 5, 7]

print(f"\n[1] Testing hyperparameter combinations:")
print(f"    - numTrees: {num_trees_options} ")
print(f"    - maxDepth: {max_depth_options} ")
print(f"    - Total combinations: {len(num_trees_options) * len(max_depth_options)}")

In [0]:
# Step 3: Grid search - test all combinations

# Evaluators already created in Step 1 above
# evaluator (F1 score) is used to measure model performance during grid search

print("\n[2] Running grid search on validation set...")
best_score = 0
best_params = {}

# nested loops to test every combination:
for num_trees in num_trees_options:
    for max_depth in max_depth_options:

        # Train model with current parameters

        # Create a Random Forest with specific hyperparameters
        rf_temp = RandomForestClassifier(
            labelCol="label",
            featuresCol="features",
            numTrees=num_trees,
            maxDepth=max_depth,
            seed=42
        )

        # train it on train data
        rf_temp_model = rf_temp.fit(train_data)
        
        # Predict on validation set
        predictions = rf_temp_model.transform(val_data)

        # Evaluate F1 score
        score = evaluator_f1.evaluate(predictions)
        
        # Track the best parameter that achieve highest F1 score
        if score > best_score:
            best_score = score
            best_params = {'numTrees': num_trees, 'maxDepth': max_depth}

print(f"\n[3] Best hyperparameters found (validation F1: {best_score:.4f}):")
print(f"    - numTrees: {best_params['numTrees']}")
print(f"    - maxDepth: {best_params['maxDepth']}")

In [0]:
# Step 4: Retrain with best parameters on full training set (100% training dataset)
print("\n[4] Retraining with best parameters on full training set...")
rf_best_model = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=best_params['numTrees'],
    maxDepth=best_params['maxDepth'],
    seed=42
).fit(train_df)

Step 5: Evaluate / Make predictions on test set

Creates 4 evaluators to measure different performance metrics:

- Accuracy: Overall % correct
- Precision: Of predicted positives, how many were correct?
- Recall: Of actual positives, how many did we find?
- F1 Score: Harmonic mean of precision and recall

In [0]:
# Step 5: Evaluate / Make predictions on test set
rf_predictions = rf_best_model.transform(test_df)

# Evaluators already created in Step 1
# No need to recreate them - just use evaluator_acc, evaluator_prec, evaluator_rec, evaluator_f1

# calculate all 4 metrics and stores them for later comparison
rf_accuracy = evaluator_acc.evaluate(rf_predictions)
rf_precision = evaluator_prec.evaluate(rf_predictions)
rf_recall = evaluator_rec.evaluate(rf_predictions)
rf_f1 = evaluator_f1.evaluate(rf_predictions)

print("\n[5] Performance on test set:")
print(f"    - Accuracy:  {rf_accuracy:.4f}")
print(f"    - Precision: {rf_precision:.4f}")
print(f"    - Recall:    {rf_recall:.4f}")
print(f"    - F1 Score:  {rf_f1:.4f}")
print("="*60)

### **Alternative: Automated Tuning with TrainValidationSplit**

Instead of manual grid search, we can use Spark MLlib's built-in **TrainValidationSplit** which automates the entire hyperparameter tuning process.

In [0]:
# SECTION 4 ALTERNATIVE: TrainValidationSplit for Random Forest
# Automates the grid search process

from pyspark.ml.tuning import TrainValidationSplit, ParamGridBuilder
from pyspark.ml.classification import RandomForestClassifier

print("="*60)
print("TRAINVALIDATIONSPLIT - RANDOM FOREST")
print("="*60)

try:
    # Step 1: Create the model
    rf = RandomForestClassifier(labelCol="label", featuresCol="features", seed=42)
    
    # Step 2: Build parameter grid (same as manual)
    paramGrid = ParamGridBuilder() \
        .addGrid(rf.numTrees, [10, 20, 30]) \
        .addGrid(rf.maxDepth, [3, 5, 7]) \
        .build()
    
    # Step 3: Create TrainValidationSplit
    tvs = TrainValidationSplit(
        estimator=rf,
        estimatorParamMaps=paramGrid,
        evaluator=evaluator_f1,  # F1 score evaluator
        trainRatio=0.8,  # 80% train, 20% validation
        seed=42
    )
    
    # Step 4: Fit (automatically tests all combinations)
    print("\n[1] Running automated grid search...")
    print(f"    - Testing {len(paramGrid)} hyperparameter combinations")
    print(f"    - Using 80/20 train/validation split")
    rf_tvs_model = tvs.fit(train_df_clean)
    
    # Step 5: Evaluate on test set
    rf_tvs_predictions = rf_tvs_model.transform(test_df_clean)
    rf_tvs_accuracy = evaluator_acc.evaluate(rf_tvs_predictions)
    rf_tvs_f1 = evaluator_f1.evaluate(rf_tvs_predictions)
    
    print("\n[2] TrainValidationSplit Results:")
    print(f"    - Accuracy:  {rf_tvs_accuracy:.4f}")
    print(f"    - F1 Score:  {rf_tvs_f1:.4f}")
    print("\n✅ Automated tuning completed!")
    print("\n🔄 Comparison: Manual vs TrainValidationSplit")
    print(f"    Manual Grid Search:      F1 = {rf_f1:.4f}")
    print(f"    TrainValidationSplit:    F1 = {rf_tvs_f1:.4f}")
    print(f"    Difference:              {abs(rf_f1 - rf_tvs_f1):.4f}")
    
except Exception as e:
    print(f"\n❌ TrainValidationSplit failed: {e}")
    print("    Ensure setup cell (column cleaning + volume) was run first")

print("="*60)

### **SECTION 5: Logistic Regression Classifier with Hyperparameter Tuning**

#### **What is Logistic Regression?**
A linear model that estimates probabilities for each class using mathematical functions.

#### **Why use Logistic Regression for the Iris dataset?**
- The Iris dataset is a classic multiclass classification problem with three well-separated classes and numeric features.
- Logistic Regression is simple, interpretable, and effective for linearly separable data, which fits the Iris dataset well.
- It provides probabilistic outputs, allowing us to understand the model's confidence in its predictions.
- The model is computationally efficient and less prone to overfitting on small datasets like Iris.
- Logistic Regression serves as a strong baseline for comparison with more complex models.

#### **Hyperparameter Tuning Approach:**
* **Manual Grid Search**: Tests all combinations of hyperparameters systematically
* **Validation Split**: Uses 80% of training data for model fitting, 20% for validation
* **Parameters tuned**: `regParam` (regularization strength) and `maxIter` (maximum iterations)
* **Selection Criterion**: F1 score on validation set

%md

 `regParam` (regularization strength) - contols the strength of regularization - penalty on large model coefficients to prevent overfitting
Range coverage:
- 0.001 = Very weak regularization (model has more freedom)
- 0.01 = Moderate regularization (balanced)
- 0.1 = Strong regularization (more penalty, simpler model)
- If too small - almost no regularization, risk of overfitting
- If too large - too much penalty. model becomes underfitting.

 `maxIter` (maximum iterations) - to find the best model weights
- 50 iterations: Quick baseline - often sufficient for simple datasets like Iris
- 100 iterations: Standard choice for most problems
- 150 iterations: Ensures convergence for more complex patterns
- If too few (<50)- model might be undertraining
- If too many (>500) - unnecessary computation time, risk of overfitting

In [0]:
# SECTION 5: Logistic Regression Classifier with Hyperparameter Tuning

from pyspark.ml.classification import LogisticRegression

print("="*60)
print("LOGISTIC REGRESSION CLASSIFIER")
print("="*60)

# Step 1: Split training data for validation (reusing from Random Forest)
# Training set (already 70% of original data) split into 80% for training, 20% for validation
# 80% is used to fit models with different hyperparameters
# 20% is used to test which hyperparameters work best
# validation set gives an unbiased way to compare different configurations
# Note: train_data and val_data already created in Section 4

# Step 2: Define hyperparameter grid
# the value is based on machine learning best practices and computational efficiency.
reg_param_options = [0.001, 0.01, 0.1]

# for small dataset like Iris, 50-150 iterations is empirically sufficient. Beyond 200 iterations rarely improves performance on simple dataset.
max_iter_options = [50, 100, 150]

print(f"\n[1] Testing hyperparameter combinations:")
print(f"    - regParam: {reg_param_options}")
print(f"    - maxIter: {max_iter_options}")
print(f"    - Total combinations: {len(reg_param_options) * len(max_iter_options)}")

In [0]:
# Step 3: Grid search - test all combinations

# this creates an evaluator to measure model performance using F1 score
# F1 score = harmonic mean of precision & recall, good for balanced evaluation
# Note: evaluator already created in Section 4

print("\n[2] Running grid search on validation set...")
best_score_lr = 0
best_params_lr = {}

# nested loops to test every combination:
for reg_param in reg_param_options:
    for max_iter in max_iter_options:

        # Train model with current parameters

        # Create a Logistic Regression with specific hyperparameters
        lr_temp = LogisticRegression(
            labelCol="label",
            featuresCol="features",
            regParam=reg_param,
            maxIter=max_iter
        )

        # train it on train data
        lr_temp_model = lr_temp.fit(train_data)
        
        # Predict on validation set
        predictions = lr_temp_model.transform(val_data)

        # Evaluate F1 score
        score = evaluator_f1.evaluate(predictions)
        
        # Track the best parameter that achieve highest F1 score
        if score > best_score_lr:
            best_score_lr = score
            best_params_lr = {'regParam': reg_param, 'maxIter': max_iter}

print(f"\n[3] Best hyperparameters found (validation F1: {best_score_lr:.4f}):")
print(f"    - regParam: {best_params_lr['regParam']}")
print(f"    - maxIter: {best_params_lr['maxIter']}")

In [0]:
# Step 4: Retrain with best parameters on full training set (100% training dataset)
print("\n[4] Retraining with best parameters on full training set...")
lr_best_model = LogisticRegression(
    labelCol="label",
    featuresCol="features",
    regParam=best_params_lr['regParam'],
    maxIter=best_params_lr['maxIter']
).fit(train_df)

Step 5: Evaluate / Make predictions on test set

Creates 4 evaluators to measure different performance metrics:

- Accuracy: Overall % correct
- Precision: Of predicted positives, how many were correct?
- Recall: Of actual positives, how many did we find?
- F1 Score: Harmonic mean of precision and recall

In [0]:
# Step 5: Evaluate / Make predictions on test set
lr_predictions = lr_best_model.transform(test_df)

# creates 4 evaluators to measure different performance metrics:
# Note: evaluators already created in Section 4 (evaluator_acc, evaluator_prec, evaluator_rec, evaluator_f1)

# calculate all 4 metrics and stores them for later comparison
lr_accuracy = evaluator_acc.evaluate(lr_predictions)
lr_precision = evaluator_prec.evaluate(lr_predictions)
lr_recall = evaluator_rec.evaluate(lr_predictions)
lr_f1 = evaluator_f1.evaluate(lr_predictions)

print("\n[5] Performance on test set:")
print(f"    - Accuracy:  {lr_accuracy:.4f}")
print(f"    - Precision: {lr_precision:.4f}")
print(f"    - Recall:    {lr_recall:.4f}")
print(f"    - F1 Score:  {lr_f1:.4f}")
print("="*60)

### **Alternative: Automated Tuning with CrossValidator**

Instead of manual grid search, we can use Spark MLlib's built-in **CrossValidator** which uses k-fold cross-validation for more robust hyperparameter selection.

In [0]:
# SECTION 5 ALTERNATIVE: CrossValidator for Logistic Regression
# Uses 3-fold cross-validation for more robust tuning

from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.classification import LogisticRegression

print("="*60)
print("CROSSVALIDATOR - LOGISTIC REGRESSION")
print("="*60)

try:
    # Step 1: Create the model
    lr = LogisticRegression(labelCol="label", featuresCol="features")
    
    # Step 2: Build parameter grid (same as manual)
    paramGrid = ParamGridBuilder() \
        .addGrid(lr.regParam, [0.001, 0.01, 0.1]) \
        .addGrid(lr.maxIter, [50, 100, 150]) \
        .build()
    
    # Step 3: Create CrossValidator (3-fold)
    crossval = CrossValidator(
        estimator=lr,
        estimatorParamMaps=paramGrid,
        evaluator=evaluator_f1,  # F1 score evaluator
        numFolds=3,  # 3-fold cross-validation
        seed=42
    )
    
    # Step 4: Fit (automatically tests all combinations with k-fold)
    print("\n[1] Running 3-fold cross-validation...")
    print(f"    - Testing {len(paramGrid)} hyperparameter combinations")
    print(f"    - Training {len(paramGrid) * 3} models total (9 × 3 folds)")
    
    lr_cv_model = crossval.fit(train_df_clean)
    
    # Step 5: Evaluate on test set
    lr_cv_predictions = lr_cv_model.transform(test_df_clean)
    lr_cv_accuracy = evaluator_acc.evaluate(lr_cv_predictions)
    lr_cv_f1 = evaluator_f1.evaluate(lr_cv_predictions)
    
    print("\n[2] CrossValidator Results:")
    print(f"    - Accuracy:  {lr_cv_accuracy:.4f}")
    print(f"    - F1 Score:  {lr_cv_f1:.4f}")

    # Step 6: Compare to manual grid search
    print("\n🔄 Comparison: Manual vs CrossValidator")
    print(f"    Manual Grid Search:      F1 = {lr_f1:.4f}")
    print(f"    CrossValidator (3-fold): F1 = {lr_cv_f1:.4f}")
    print(f"    Difference:              {abs(lr_f1 - lr_cv_f1):.4f}")
    
except Exception as e:
    print(f"\n❌ CrossValidator failed: {e}")
    print("    Ensure setup cell (column cleaning + volume) was run first")

print("="*60)

### **SECTION 6: Decision Tree Classifier with Hyperparameter Tuning**

#### **What is Decision Tree?**
A single tree that makes decisions by splitting data based on feature values in a flowchart-like structure.

#### **Why use Decision Tree for the Iris dataset?**
- The Iris dataset has clear, well-separated classes, making it ideal for decision boundaries created by trees.
- Decision Trees are easy to interpret and visualize, helping to understand which features are important for classification.
- They handle both numerical and categorical features and require minimal data preparation.
- Decision Trees can capture non-linear relationships, which may exist in the Iris data.
- They serve as a strong, interpretable baseline for comparison with more complex models.

#### **Hyperparameter Tuning Approach:**
* **Manual Grid Search**: Tests all combinations of hyperparameters systematically
* **Validation Split**: Uses 80% of training data for model fitting, 20% for validation
* **Parameters tuned**: `maxDepth` (tree depth limit) and `minInstancesPerNode` (minimum samples per node)
* **Selection Criterion**: F1 score on validation set

In [0]:
# SECTION 6: Decision Tree Classifier with Hyperparameter Tuning

from pyspark.ml.classification import DecisionTreeClassifier

print("="*60)
print("DECISION TREE CLASSIFIER")
print("="*60)

# Step 1: Split training data for validation (reusing from Random Forest)
# Training set (already 70% of original data) split into 80% for training, 20% for validation
# 80% is used to fit models with different hyperparameters
# 20% is used to test which hyperparameters work best
# validation set gives an unbiased way to compare different configurations
# Note: train_data and val_data already created in Section 4

# Step 2: Define hyperparameter grid
max_depth_options = [3, 5, 7, 10]
min_instances_options = [1, 5, 10]

print(f"\n[1] Testing hyperparameter combinations:")
print(f"    - maxDepth: {max_depth_options}")
print(f"    - minInstancesPerNode: {min_instances_options}")
print(f"    - Total combinations: {len(max_depth_options) * len(min_instances_options)}")

In [0]:
# Step 3: Grid search - test all combinations

# this creates an evaluator to measure model performance using F1 score
# F1 score = harmonic mean of precision & recall, good for balanced evaluation
# Note: evaluator already created in Section 4

print("\n[2] Running grid search on validation set...")
best_score_dt = 0
best_params_dt = {}

# nested loops to test every combination:
for max_depth in max_depth_options:
    for min_instances in min_instances_options:

        # Train model with current parameters

        # Create a Decision Tree with specific hyperparameters
        dt_temp = DecisionTreeClassifier(
            labelCol="label",
            featuresCol="features",
            maxDepth=max_depth,
            minInstancesPerNode=min_instances,
            seed=42
        )

        # train it on train data
        dt_temp_model = dt_temp.fit(train_data)
        
        # Predict on validation set
        predictions = dt_temp_model.transform(val_data)

        # Evaluate F1 score
        score = evaluator_f1.evaluate(predictions)
        
        # Track the best parameter that achieve highest F1 score
        if score > best_score_dt:
            best_score_dt = score
            best_params_dt = {'maxDepth': max_depth, 'minInstancesPerNode': min_instances}

print(f"\n[3] Best hyperparameters found (validation F1: {best_score_dt:.4f}):")
print(f"    - maxDepth: {best_params_dt['maxDepth']}")
print(f"    - minInstancesPerNode: {best_params_dt['minInstancesPerNode']}")

In [0]:
# Step 4: Retrain with best parameters on full training set (100% training dataset)
print("\n[4] Retraining with best parameters on full training set...")
dt_best_model = DecisionTreeClassifier(
    labelCol="label",
    featuresCol="features",
    maxDepth=best_params_dt['maxDepth'],
    minInstancesPerNode=best_params_dt['minInstancesPerNode'],
    seed=42
).fit(train_df)

Step 5: Evaluate / Make predictions on test set

Creates 4 evaluators to measure different performance metrics:

- Accuracy: Overall % correct
- Precision: Of predicted positives, how many were correct?
- Recall: Of actual positives, how many did we find?
- F1 Score: Harmonic mean of precision and recall

In [0]:
# Step 5: Evaluate / Make predictions on test set
dt_predictions = dt_best_model.transform(test_df)

# creates 4 evaluators to measure different performance metrics:
# Note: evaluators already created in Section 4 (evaluator_acc, evaluator_prec, evaluator_rec, evaluator_f1)

# calculate all 4 metrics and stores them for later comparison
dt_accuracy = evaluator_acc.evaluate(dt_predictions)
dt_precision = evaluator_prec.evaluate(dt_predictions)
dt_recall = evaluator_rec.evaluate(dt_predictions)
dt_f1 = evaluator_f1.evaluate(dt_predictions)

print("\n[5] Performance on test set:")
print(f"    - Accuracy:  {dt_accuracy:.4f}")
print(f"    - Precision: {dt_precision:.4f}")
print(f"    - Recall:    {dt_recall:.4f}")
print(f"    - F1 Score:  {dt_f1:.4f}")
print("="*60)

### **Alternative: Automated Tuning with TrainValidationSplit**

Instead of manual grid search, we can use Spark MLlib's built-in **TrainValidationSplit** which automates the entire hyperparameter tuning process.

In [0]:
# SECTION 6 ALTERNATIVE: TrainValidationSplit for Decision Tree
# Automates the grid search process

from pyspark.ml.tuning import TrainValidationSplit, ParamGridBuilder
from pyspark.ml.classification import DecisionTreeClassifier

print("="*60)
print("TRAINVALIDATIONSPLIT - DECISION TREE")
print("="*60)

try:
    # Step 1: Create the model
    dt = DecisionTreeClassifier(labelCol="label", featuresCol="features", seed=42)
    
    # Step 2: Build parameter grid (same as manual)
    paramGrid = ParamGridBuilder() \
        .addGrid(dt.maxDepth, [3, 5, 7, 10]) \
        .addGrid(dt.minInstancesPerNode, [1, 5, 10]) \
        .build()
    
    # Step 3: Create TrainValidationSplit
    tvs = TrainValidationSplit(
        estimator=dt,
        estimatorParamMaps=paramGrid,
        evaluator=evaluator_f1,  # F1 score evaluator
        trainRatio=0.8,  # 80% train, 20% validation
        seed=42
    )
    
    # Step 4: Fit (automatically tests all combinations)
    print("\n[1] Running automated grid search...")
    print(f"    - Testing {len(paramGrid)} hyperparameter combinations")
    print(f"    - Using 80/20 train/validation split")
    
    dt_tvs_model = tvs.fit(train_df_clean)
    
    # Step 5: Evaluate on test set
    dt_tvs_predictions = dt_tvs_model.transform(test_df_clean)
    dt_tvs_accuracy = evaluator_acc.evaluate(dt_tvs_predictions)
    dt_tvs_f1 = evaluator_f1.evaluate(dt_tvs_predictions)
    
    print("\n[2] TrainValidationSplit Results:")
    print(f"    - Accuracy:  {dt_tvs_accuracy:.4f}")
    print(f"    - F1 Score:  {dt_tvs_f1:.4f}")
    print("\n🔄 Comparison: Manual vs TrainValidationSplit")
    print(f"    Manual Grid Search:      F1 = {dt_f1:.4f}")
    print(f"    TrainValidationSplit:    F1 = {dt_tvs_f1:.4f}")
    print(f"    Difference:              {abs(dt_f1 - dt_tvs_f1):.4f}")
    
except Exception as e:
    print(f"\n❌ TrainValidationSplit failed: {e}")
    print("    Ensure setup cell (column cleaning + volume) was run first")

print("="*60)

### **SECTION 7: Multilayer Perceptron Classifier with Hyperparameter Tuning**

#### **What is Multilayer Perceptron?**
##### A feedforward artificial neural network with multiple layers that can learn complex non-linear patterns through backpropagation

#### **Why use Multilayer Perceptron for the Iris dataset?**
- The Iris dataset may contain non-linear relationships that simpler models (like logistic regression) cannot capture.
- MLPs can model complex decision boundaries, improving performance when classes are not perfectly linearly separable.
- They provide a flexible architecture that can adapt to the complexity of the data.
- MLPs serve as a strong benchmark for comparing traditional machine learning models with neural networks.

#### **Hyperparameter Tuning Approach:**
* **Manual Grid Search**: Tests different network architectures
* **Validation Split**: Uses 80% of training data for model fitting, 20% for validation
* **Parameter tuned**: `layers` (network architecture: input, hidden layers, output)
* **Selection Criterion**: F1 score on validation set
* **Note**: Input layer size = 4 (features), Output layer size = 3 (classes).

In [0]:
# SECTION 7: Multilayer Perceptron Classifier with Hyperparameter Tuning

from pyspark.ml.classification import MultilayerPerceptronClassifier

print("="*60)
print("MULTILAYER PERCEPTRON CLASSIFIER")
print("="*60)

# Step 1: Split training data for validation (reusing from Random Forest)
# Training set (already 70% of original data) split into 80% for training, 20% for validation
# 80% is used to fit models with different hyperparameters
# 20% is used to test which hyperparameters work best
# validation set gives an unbiased way to compare different configurations
# Note: train_data and val_data already created in Section 4

# Step 2: Define hyperparameter grid (network architectures)
# Format: [input_layer, hidden_layer1, hidden_layer2, ..., output_layer]
# Input = 4 features, Output = 3 classes
layers_options = [
    [4, 5, 3],      # 1 hidden layer with 5 neurons
    [4, 10, 3],     # 1 hidden layer with 10 neurons
    [4, 5, 5, 3],   # 2 hidden layers with 5 neurons each
    [4, 10, 5, 3]   # 2 hidden layers with 10 and 5 neurons
]

print(f"\n[1] Testing hyperparameter combinations:")
print(f"    - Network architectures (layers): {len(layers_options)} options")
for i, layers in enumerate(layers_options, 1):
    print(f"      {i}. {layers}")

In [0]:
# Step 3: Grid search - test all combinations

# this creates an evaluator to measure model performance using F1 score
# F1 score = harmonic mean of precision & recall, good for balanced evaluation
# Note: evaluator already created in Section 4

print("\n[2] Running grid search on validation set...")
best_score_mlp = 0
best_params_mlp = {}

# loop to test every combination:
for layers in layers_options:

    # Train model with current parameters

    # Create a Multilayer Perceptron with specific hyperparameters
    mlp_temp = MultilayerPerceptronClassifier(
        labelCol="label",
        featuresCol="features",
        layers=layers,
        maxIter=100,
        seed=42
    )

    # train it on train data
    mlp_temp_model = mlp_temp.fit(train_data)
    
    # Predict on validation set
    predictions = mlp_temp_model.transform(val_data)

    # Evaluate F1 score
    score = evaluator_f1.evaluate(predictions)
    
    # Track the best parameter that achieve highest F1 score
    if score > best_score_mlp:
        best_score_mlp = score
        best_params_mlp = {'layers': layers}

print(f"\n[3] Best hyperparameters found (validation F1: {best_score_mlp:.4f}):")
print(f"    - layers: {best_params_mlp['layers']}")

In [0]:
# Step 4: Retrain with best parameters on full training set (100% training dataset)
print("\n[4] Retraining with best parameters on full training set...")
mlp_best_model = MultilayerPerceptronClassifier(
    labelCol="label",
    featuresCol="features",
    layers=best_params_mlp['layers'],
    maxIter=100,
    seed=42
).fit(train_df)

In [0]:
# Step 5: Evaluate / Make predictions on test set
mlp_predictions = mlp_best_model.transform(test_df)

# creates 4 evaluators to measure different performance metrics:
# Note: evaluators already created in Section 4 (evaluator_acc, evaluator_prec, evaluator_rec, evaluator_f1)

# calculate all 4 metrics and stores them for later comparison
mlp_accuracy = evaluator_acc.evaluate(mlp_predictions)
mlp_precision = evaluator_prec.evaluate(mlp_predictions)
mlp_recall = evaluator_rec.evaluate(mlp_predictions)
mlp_f1 = evaluator_f1.evaluate(mlp_predictions)

print("\n[5] Performance on test set:")
print(f"    - Accuracy:  {mlp_accuracy:.4f}")
print(f"    - Precision: {mlp_precision:.4f}")
print(f"    - Recall:    {mlp_recall:.4f}")
print(f"    - F1 Score:  {mlp_f1:.4f}")
print("="*60)

### **Alternative: Automated Tuning with CrossValidator**

Instead of manual grid search, we can use Spark MLlib's built-in **CrossValidator** which uses k-fold cross-validation for more robust hyperparameter selection.

In [0]:
# SECTION 7 ALTERNATIVE: CrossValidator for Multilayer Perceptron
# Uses 3-fold cross-validation for more robust tuning

from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.classification import MultilayerPerceptronClassifier

print("="*60)
print("CROSSVALIDATOR - MULTILAYER PERCEPTRON")
print("="*60)

try:
    # Step 1: Create the model
    mlp = MultilayerPerceptronClassifier(labelCol="label", featuresCol="features", maxIter=100, seed=42)
    
    # Step 2: Build parameter grid (same as manual)
    paramGrid = ParamGridBuilder() \
        .addGrid(mlp.layers, [[4, 5, 3], [4, 10, 3], [4, 5, 5, 3], [4, 10, 5, 3]]) \
        .build()
    
    # Step 3: Create CrossValidator (3-fold)
    crossval = CrossValidator(
        estimator=mlp,
        estimatorParamMaps=paramGrid,
        evaluator=evaluator_f1,  # F1 score evaluator
        numFolds=3,  # 3-fold cross-validation
        seed=42
    )
    
    # Step 4: Fit (automatically tests all combinations with k-fold)
    print("\n[1] Running 3-fold cross-validation...")
    print(f"    - Testing {len(paramGrid)} network architectures")
    print(f"    - Training {len(paramGrid) * 3} models total (4 × 3 folds)")
    
    mlp_cv_model = crossval.fit(train_df_clean)
    
    # Step 5: Evaluate on test set
    mlp_cv_predictions = mlp_cv_model.transform(test_df_clean)
    mlp_cv_accuracy = evaluator_acc.evaluate(mlp_cv_predictions)
    mlp_cv_f1 = evaluator_f1.evaluate(mlp_cv_predictions)
    
    print("\n[2] CrossValidator Results:")
    print(f"    - Accuracy:  {mlp_cv_accuracy:.4f}")
    print(f"    - F1 Score:  {mlp_cv_f1:.4f}")
    print("\n✅ Automated k-fold cross-validation completed!")
    print("\n🔄 Comparison: Manual vs CrossValidator")
    print(f"    Manual Grid Search:      F1 = {mlp_f1:.4f}")
    print(f"    CrossValidator (3-fold): F1 = {mlp_cv_f1:.4f}")
    print(f"    Difference:              {abs(mlp_f1 - mlp_cv_f1):.4f}")
    
except Exception as e:
    print(f"\n❌ CrossValidator failed: {e}")
    print("    Ensure setup cell (column cleaning + volume) was run first")

print("="*60)

### **SECTION 8: Comparative Analysis of Classification Models**

#### **Experimental Design**

This section compares four classification algorithms on the Iris dataset using two hyperparameter tuning methodologies:

**Manual Grid Search** - Exhaustive search over predefined hyperparameter combinations using an 80/20 train-validation split. Best parameters selected based on validation F1 score, then model retrained on full training set.

**Automated Tuning** - Leverages Spark MLlib's built-in optimization:
* Random Forest & Decision Tree: TrainValidationSplit (automated 80/20 split)
* Logistic Regression & Multilayer Perceptron: CrossValidator (3-fold cross-validation)

Both approaches evaluate performance on a held-out test set (30% of data, 51 samples) to ensure unbiased performance estimation.

---

The following tables present quantitative results, followed by detailed analysis and discussion.

In [0]:
# SECTION 8: Comparative Analysis of Classification Models
# Part A: Manual Grid Search Results

import pandas as pd

print("\n" + "="*70)
print("TABLE 1: Performance Metrics - Manual Grid Search")
print("="*70)

# Compile performance metrics from manual hyperparameter tuning
results = pd.DataFrame([
    {'Model': 'Random Forest', 'Accuracy': rf_accuracy, 'Precision': rf_precision, 'Recall': rf_recall, 'F1 Score': rf_f1},
    {'Model': 'Logistic Regression', 'Accuracy': lr_accuracy, 'Precision': lr_precision, 'Recall': lr_recall, 'F1 Score': lr_f1},
    {'Model': 'Decision Tree', 'Accuracy': dt_accuracy, 'Precision': dt_precision, 'Recall': dt_recall, 'F1 Score': dt_f1},
    {'Model': 'Multilayer Perceptron', 'Accuracy': mlp_accuracy, 'Precision': mlp_precision, 'Recall': mlp_recall, 'F1 Score': mlp_f1}
])

# Rank models by F1 Score
results = results.sort_values('F1 Score', ascending=False).reset_index(drop=True)
results.insert(0, 'Rank', range(1, len(results) + 1))

print("\n" + results.to_string(index=False))

# Statistical summary
print("\n" + "="*70)
print("Summary Statistics (Manual Tuning):")
print(f"  Top Performer:         {results.iloc[0]['Model']} (F1 = {results.iloc[0]['F1 Score']:.4f})")
print(f"  Mean F1 Score:         {results['F1 Score'].mean():.4f}")
print(f"  Standard Deviation:    {results['F1 Score'].std():.4f}")
print(f"  Performance Range:     {results['F1 Score'].max() - results['F1 Score'].min():.4f}")
print(f"  Models with F1 > 0.96: {sum(results['F1 Score'] > 0.96)}/4")
print("="*70)

In [0]:
# SECTION 8: Comparative Analysis of Classification Models
# Part B: Automated Hyperparameter Tuning Results

import pandas as pd

print("\n" + "="*70)
print("TABLE 2: Performance Metrics - Automated Tuning")
print("="*70)

# Compile performance metrics from automated hyperparameter tuning
results_auto = pd.DataFrame([
    {'Model': 'Random Forest', 'Method': 'TrainValidationSplit', 'Accuracy': rf_tvs_accuracy, 'F1 Score': rf_tvs_f1},
    {'Model': 'Logistic Regression', 'Method': 'CrossValidator (3-fold)', 'Accuracy': lr_cv_accuracy, 'F1 Score': lr_cv_f1},
    {'Model': 'Decision Tree', 'Method': 'TrainValidationSplit', 'Accuracy': dt_tvs_accuracy, 'F1 Score': dt_tvs_f1},
    {'Model': 'Multilayer Perceptron', 'Method': 'CrossValidator (3-fold)', 'Accuracy': mlp_cv_accuracy, 'F1 Score': mlp_cv_f1}
])

# Rank models by F1 Score
results_auto = results_auto.sort_values('F1 Score', ascending=False).reset_index(drop=True)
results_auto.insert(0, 'Rank', range(1, len(results_auto) + 1))

print("\n" + results_auto.to_string(index=False))

# Statistical summary
print("\n" + "="*70)
print("Summary Statistics (Automated Tuning):")
print(f"  Top Performer:         {results_auto.iloc[0]['Model']} (F1 = {results_auto.iloc[0]['F1 Score']:.4f})")
print(f"  Mean F1 Score:         {results_auto['F1 Score'].mean():.4f}")
print(f"  Standard Deviation:    {results_auto['F1 Score'].std():.4f}")
print(f"  Performance Range:     {results_auto['F1 Score'].max() - results_auto['F1 Score'].min():.4f}")
print(f"  Uniform Performance:   {'Yes' if results_auto['F1 Score'].std() < 0.0001 else 'No'}")
print("="*70)

### **8.3 Discussion and Analysis**

---

#### **Comparative Performance Observations**

Table 1 shows that manual grid search produced F1 scores ranging from 0.9406 to 0.9802 across the four models. Logistic Regression and Multilayer Perceptron tied for the best performance at 0.9802, while Decision Tree scored lowest at 0.9406. The standard deviation of 0.0189 indicates moderate performance variation between models.

Table 2 shows the uniform performance. Sll four models achieved identical F1 scores of 0.9802 under automated tuning (standard deviation = 0.0000). This convergence is particularly notable for Decision Tree, which improved by 3.96 percentage points from its manual tuning result.

#### **Automated Tuning vs. Manual Search**

The automated approaches have several advantages over manual grid search:

- CrossValidator tests each hyperparameter combination across three different train-validation splits rather than relying on a single split. This reduces the variance in hyperparameter selection. It makes the results more robust. In our manual approach, the best hyperparameters were selected based on performance on one specific validation set. This may not represent the optimal configuration for the overall data distribution.

- The automated methods likely explored the hyperparameter space more efficiently. While manual grid tested reasonable ranges, the automated tuning may have identified subtle parameter combinations that work particularly well for this dataset. For instance, Decision Tree's dramatic improvement suggests the automated method found a better maxDepth-minInstancesPerNode balance than  manual grid provided.



#### **Model Selection Considerations**

Despite equivalent F1 scores under automated tuning, the models differ in important practical ways:

**Logistic Regression** achieved top performance with the simplest architecture. For production deployment, this would be the preferred choice. The model is interpretable (coefficients show feature importance), computationally inexpensive, and easy to maintain. There's no practical benefit to using a more complex model when a linear classifier achieves equivalent accuracy.

**Multilayer Perceptron** matched Logistic Regression's performance but requires more computational resources for training and inference. The fact that MLP didn't outperform the linear model confirms that the Iris dataset doesn't contain non-linear patterns that require neural network capabilities. However, MLP still serves a useful purpose. If the simpler model performs just as well, it validates that we're not missing complex patterns in the data.

**Random Forest** improved from 0.9608 to 0.9802 under automated tuning. The ensemble method's strength is variance reduction through averaging multiple trees, but this benefit is less pronounced on small datasets. With only 99 training samples, each tree sees a limited subset of data through bootstrap sampling, which can actually increase variance rather than reduce it if not tuned properly.

**Decision Tree** showed the largest improvement (0.9406 to 0.9802) between tuning methods. Single decision trees are highly sensitive to hyperparameter choices - too deep and they overfit to training noise, too shallow and they miss important patterns. The automated tuning clearly found a better bias-variance tradeoff than the manual grid search.



#### **Limitations**

Several limitations should be noted:

- The test set of 51 samples provides limited granularity for evaluating small performance differences. With this sample size, statistical significance testing would be needed to determine if apparent performance differences are meaningful or just noise.

- We used a single 70-30 train-test split for final evaluation. While cross-validation was employed during hyperparameter tuning, the final performance estimates are based on one test set. Repeated k-fold evaluation of the tuned models would provide confidence intervals around these point estimates.

- The Iris dataset is small and relatively simple compared to real-world classification problems. These results may not generalize to larger, more complex datasets. The algorithm choice and tuning methodology might have greater impact on larger dataset.


#### **Recommendations**

Based on this analysis, for similar small-scale multiclass classification problems:

1. Start with Logistic Regression as a baseline. If it performs well, prefer it over more complex models for production deployment.

2. Use CrossValidator over manual grid search when possible. The k-fold approach provides more reliable hyperparameter selection with minimal additional code.

3. For Decision Trees and Random Forests, automated tuning is particularly valuable given their sensitivity to hyperparameter choices.

4. When models achieve similar performance, choose based on interpretability, computational cost, and maintenance considerations rather than minor differences in accuracy.

### **SECTION 9: Model Predictions and Error Analysis**

#### **Purpose**

While aggregate metrics like F1 score provide overall performance assessment, examining individual predictions reveals how the model behaves on specific samples. This section presents predictions from the top-performing model (Multilayer Perceptron, F1 = 0.9802) to:

* Verify the model makes sensible predictions on actual test data
* Assess prediction confidence through probability distributions  
* Identify any systematic misclassification patterns
* Validate that high aggregate performance translates to correct individual decisions

#### **Output Format**

Each prediction includes:
* **features**: Vector of four measurements [sepal length, sepal width, petal length, petal width] in cm
* **label**: True class (0=Setosa, 1=Versicolor, 2=Virginica)  
* **prediction**: Model's predicted class
* **probability**: Confidence distribution across the three classes

A well-calibrated classifier should show high probability (>0.90) for the predicted class when correct, with low probabilities for alternative classes.

In [0]:
# SECTION 9: Model Predictions and Error Analysis

from pyspark.sql.functions import col

print("="*70)
print("TABLE 3: Sample Predictions from Top-Performing Model")
print("Model: Multilayer Perceptron (Manual Tuning, F1 = 0.9802)")
print("="*70)
print("\nDisplaying 20 samples from test set (total test samples: 51)")
print("Feature vector: [sepal_length, sepal_width, petal_length, petal_width]")
print("Class labels: 0=Setosa, 1=Versicolor, 2=Virginica\n")

# Select predictions to display
prediction_sample = mlp_predictions.select(
    "features", 
    "label", 
    "prediction", 
    "probability"
).limit(20)

display(prediction_sample)

# Calculate accuracy on displayed sample
correct = prediction_sample.filter(col("label") == col("prediction")).count()
total = prediction_sample.count()
incorrect = total - correct

print("\n" + "="*70)
print("Prediction Summary:")
print(f"  Correct classifications:    {correct}/{total} ({correct/total:.1%})")
print(f"  Incorrect classifications:  {incorrect}/{total} ({incorrect/total:.1%})")
print(f"  Full test set F1 score:     {mlp_f1:.4f}")
print("="*70)

if incorrect > 0:
    print("\nNote: Misclassifications are expected even with high F1 scores.")
    print("Examine probability vectors for misclassified samples to assess model confidence.")
else:
    print("\nNote: Perfect accuracy on this sample. Probability vectors show model confidence.")
    print("High probability (>0.9) on correct class indicates strong certainty.")